## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('../scripts')

import torch
import pytorch_lightning as pl
import seisbench.models as sbm
import yaml
from pathlib import Path

from model import PhaseNetLightning
from data_module import PhaseNetDataModule

print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Lightning version: {pl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load Configuration

In [ ]:
# Load config
config_path = Path('../configs/train_config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Model: {config['model']['name']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Max epochs: {config['training']['max_epochs']}")
print(f"  Learning rate: {config['training']['learning_rate']}")

## 3. Explore Pre-trained Models

In [ ]:
# List available pretrained models
print("Available PhaseNet pretrained models:")
available_models = sbm.PhaseNet.list_pretrained()
for model in available_models:
    print(f"  - {model}")

## 4. Load a Pre-trained Model

In [ ]:
# Load pretrained model (e.g., 'stead' model trained on STEAD dataset)
pretrained_model = sbm.PhaseNet.from_pretrained('stead')

print(f"Model loaded: {pretrained_model.__class__.__name__}")
print(f"Input channels: {pretrained_model.in_channels}")
print(f"Output classes: {pretrained_model.classes}")
print(f"Phases: {pretrained_model.labels}")
print(f"Sampling rate: {pretrained_model.sampling_rate} Hz")

## 5. Test Model on Random Data

In [ ]:
# Create random input (batch_size, channels, samples)
batch_size = 4
n_samples = 3001  # 30 seconds at 100 Hz
x = torch.randn(batch_size, 3, n_samples)

print(f"Input shape: {x.shape}")

# Forward pass
pretrained_model.eval()
with torch.no_grad():
    output = pretrained_model(x)

print(f"Output shape: {output.shape}")
print(f"Output channels: Noise, P-wave, S-wave")
print(f"Output range: [{output.min():.3f}, {output.max():.3f}]")

## 6. Visualize Model Architecture

In [ ]:
# Print model architecture
print("PhaseNet Architecture:")
print("="*50)
print(pretrained_model)
print("="*50)

# Count parameters
total_params = sum(p.numel() for p in pretrained_model.parameters())
trainable_params = sum(p.numel() for p in pretrained_model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 7. Initialize Lightning Module

In [ ]:
# Create Lightning module
pl_model = PhaseNetLightning(config)

print("PhaseNet Lightning module initialized")
print(f"Learning rate: {pl_model.learning_rate}")

## 8. Prepare Data (Example)

**Note:** You need to implement data loading based on your specific format.

In [ ]:
# This is a placeholder - implement based on your data format
# data_module = PhaseNetDataModule(
#     config=config,
#     batch_size=config['training']['batch_size'],
#     num_workers=config['training']['num_workers']
# )

print("Data module setup: Implement based on your data format")
print("See scripts/data_module.py for details")

## 9. Training (Example)

Once data is prepared, training can be started:

In [ ]:
# Example training code (uncomment when data is ready)
"""
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

# Setup callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath='../checkpoints',
    filename='phasenet-{epoch:02d}-{val_loss:.2f}',
    monitor='val_loss',
    mode='min',
    save_top_k=3
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min'
)

# Setup logger
logger = TensorBoardLogger('../results', name='phasenet_training')

# Create trainer
trainer = pl.Trainer(
    max_epochs=config['training']['max_epochs'],
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback, early_stop],
    logger=logger
)

# Train
trainer.fit(pl_model, data_module)
"""

print("Training code example provided above")

## 10. Model Inference Example

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create synthetic waveform
t = np.linspace(0, 30, 3001)  # 30 seconds
synthetic_wave = np.random.randn(3, 3001) * 0.1

# Add synthetic P and S arrivals
p_time = 1000  # 10 seconds
s_time = 1500  # 15 seconds

# Add P-wave
synthetic_wave[:, p_time:p_time+200] += np.random.randn(3, 200) * 0.5
# Add S-wave
synthetic_wave[:, s_time:s_time+300] += np.random.randn(3, 300) * 0.8

# Convert to tensor
x_inference = torch.from_numpy(synthetic_wave).float().unsqueeze(0)

# Run inference
pretrained_model.eval()
with torch.no_grad():
    predictions = pretrained_model(x_inference)

predictions_np = predictions.squeeze(0).numpy()

# Plot results
fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)

# Plot waveforms
for i, label in enumerate(['Z', 'N', 'E']):
    axes[0].plot(t, synthetic_wave[i], label=label, alpha=0.7)
axes[0].set_ylabel('Amplitude')
axes[0].legend(loc='upper right')
axes[0].set_title('Synthetic Waveforms')
axes[0].axvline(t[p_time], color='b', linestyle='--', alpha=0.5, label='True P')
axes[0].axvline(t[s_time], color='r', linestyle='--', alpha=0.5, label='True S')

# Plot predictions
axes[1].plot(t, predictions_np[0], 'g-', label='Noise', linewidth=2)
axes[1].set_ylabel('Probability')
axes[1].legend()
axes[1].set_ylim([0, 1])

axes[2].plot(t, predictions_np[1], 'b-', label='P-wave', linewidth=2)
axes[2].set_ylabel('Probability')
axes[2].legend()
axes[2].set_ylim([0, 1])

axes[3].plot(t, predictions_np[2], 'r-', label='S-wave', linewidth=2)
axes[3].set_ylabel('Probability')
axes[3].set_xlabel('Time (s)')
axes[3].legend()
axes[3].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('../results/inference_example.png', dpi=150, bbox_inches='tight')
plt.show()

print("Inference completed and results plotted")

## Summary

This notebook demonstrates:
1. Loading pretrained PhaseNet models from SeisBench
2. Model architecture and parameters
3. Basic inference on synthetic data
4. Framework structure for training

Next steps:
- Implement data loading for your specific format
- Configure training parameters
- Run training with `python scripts/train.py`
- Evaluate results and fine-tune